# Task 1: Data Audit & Exploration

This notebook reads only the configured `attrition_dataset` worksheet. It does not inspect other sheets, open or follow workbook links, or read the planted answer key. Place the provided workbook in `data/raw/` first.

The notebook shares audit logic with `task1_data_audit/scripts/run_audit.py` via `task1_data_audit/src/audit/audit_utils.py`. It does not change the source workbook or automatically clean records. No employee-level records are printed.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / 'task1_data_audit' / 'configs' / 'task1_config.json').exists():
        break
    ROOT = ROOT.parent
TASK_SRC = ROOT / 'task1_data_audit' / 'src'
if str(TASK_SRC) not in sys.path:
    sys.path.insert(0, str(TASK_SRC))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from audit.audit_utils import (
    build_column_profile,
    build_leakage_register,
    candidate_columns,
    class_distribution,
    duplicate_summary,
    missingness_summary,
    outlier_comparison,
    resolve_target_column,
)
CONFIG_PATH = ROOT / 'task1_data_audit' / 'configs' / 'task1_config.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
DATA_PATH = ROOT / config['dataset_file']
SHEET_NAME = config['sheet_name']
OUTPUT_DIR = ROOT / config.get('output_dir', 'reports/generated')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Input exists:', DATA_PATH.exists())
print('Configured sheet:', SHEET_NAME)
print('Output directory:', OUTPUT_DIR)


## Load the configured sheet only

If the input file is missing, add it to `data/raw/` or update the config. If the source column names differ from the candidates in the config, verify them against the data dictionary before updating the config. Row-level records are never printed.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Place the workbook at {DATA_PATH} or update {CONFIG_PATH}')
# Explicit single-sheet read; hyperlinks are not followed and other sheets are not inspected.
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
print(f'Rows: {len(df):,} | Columns: {df.shape[1]:,}')
print('Columns:', list(df.columns.astype(str)))

## 1. Column types and data dictionary

Shared helper `build_column_profile` assigns provisional roles. Use the blank definition field to write each column's meaning in your own words. Do not infer a business definition from a column name alone.

In [ ]:
# Same helper used by task1_data_audit/scripts/run_audit.py (no conflicting reimplementation).
data_dictionary = build_column_profile(df, config)
data_dictionary

## 2. Missingness

A percentage is a signal, not an automatic drop rule. Add the reasoned drop / fill / flag / investigate decision for every column with missing values.

In [ ]:
# Same helper used by task1_data_audit/scripts/run_audit.py.
missingness = missingness_summary(df)
missingness.head(30)

In [ ]:
plot_data = missingness.loc[missingness['missing_count'] > 0].head(20).sort_values('missing_pct')
if not plot_data.empty:
    ax = plot_data.plot.barh(x='column', y='missing_pct', legend=False, figsize=(10, max(4, len(plot_data) * 0.3)))
    ax.set_xlabel('Missing values (%)')
    ax.set_title('Columns with the most missing values (top 20)')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values detected.')

## 3. Outlier comparison: IQR and z-score

Review salary, hours, and tenure columns listed in the config. Flags are not automatically errors, particularly when the distribution is skewed. Verify units, valid ranges, and business context. Uses `outlier_comparison` from `audit_utils`, identical to the script.

In [ ]:
import pandas as pd
outlier_rows = []
for name in candidate_columns(df.columns, config.get('outlier_columns', [])):
    if pd.api.types.is_numeric_dtype(df[name]) or pd.to_numeric(df[name], errors='coerce').notna().any():
        outlier_rows.append(outlier_comparison(df[name], name))
outlier_comparison_df = pd.DataFrame(outlier_rows)
outlier_comparison_df

## 4. Target class balance

The target column and positive label semantics must be confirmed from the source specification. This cell reports class counts for the first configured target candidate that exists; do not guess the meaning of numeric or textual labels.

In [ ]:
# Same resolution + counting logic as task1_data_audit/scripts/run_audit.py.
target_col = resolve_target_column(df, config)
if target_col is None:
    print('No target candidate found. Check the column list and update the config after confirming the true target.')
    target_distribution = None
    positive_rate = None
else:
    target_distribution, positive_rate = class_distribution(df[target_col], config.get('positive_target_values', []))
    print('Target column:', target_col, '| positive rate:', positive_rate)
    target_distribution

## 5. Leakage and duplicates review

The full per-column register is built with `build_leakage_register` (same as the script). Decide based on whether the value would be known at the intended prediction date and whether it reveals the outcome. Do not remove data automatically. Identifiers are privacy exclusions, distinct from leakage.

In [ ]:
# Same helpers used by task1_data_audit/scripts/run_audit.py; covers every column, not just candidates.
leakage_register = build_leakage_register(list(df.columns), config, target_column=target_col)
leakage_register
print('Exact duplicate rows:', int(df.duplicated().sum()))
print(duplicate_summary(df, candidate_columns(df.columns, config.get('id_candidate_columns', [])), exclude_columns=['row_id']))


## 6. Save initial audit artifacts

Review the generated files manually, fill the decision/definition fields, and write the final plain-language report. This export saves summaries only, not employee-level records. For the full audited outputs (figures, decision register, final report), run `python task1_data_audit/scripts/run_audit.py`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
data_dictionary.to_csv(OUTPUT_DIR / 'data_dictionary.csv', index=False)
missingness.loc[missingness['missing_count'] > 0].to_csv(OUTPUT_DIR / 'missingness.csv', index=False)
outlier_comparison_df.to_csv(OUTPUT_DIR / 'outlier_comparison.csv', index=False)
if target_col is not None:
    target_distribution.to_csv(OUTPUT_DIR / 'target_distribution.csv', index=False)
leakage_register.to_csv(OUTPUT_DIR / 'leakage_review.csv', index=False)
print('Saved initial summaries to', OUTPUT_DIR)
print('For figures, decision register, and the final report, run: python task1_data_audit/scripts/run_audit.py')